# Liu2024 — Paper-Faithful TWFB-DGFMDM (19 bands × 7 windows + real LTSA), leaky vs honest

**What this is.** A faithful reproduction of the **TWFB-DGFMDM method exactly as Table 3 of the Liu2024 paper
describes it**, so we can confirm the leakage finding is not an artifact of the simplified 8-band code:

1. **7 time windows** (0–1, 0.5–1.5, … 3–4 s) × **19 overlapping 4-Hz bands** (8–12, 9–13, … 26–30 Hz) = 133 views.
2. Per view → per-trial **covariance** (Step 3).
3. **Real LTSA** (`sklearn LocallyLinearEmbedding(method="ltsa")`) on the Riemannian tangent vectors (Step 4).
4. **Discriminant + nearest-mean** classifier on the reduced features (Step 5); `fgmdm` available as the alternative reading.

**The experiment = the selection scope.** The pipeline is identical in all modes; only *how the (window, band) view
is chosen* changes:
- `leaky`  — view chosen by accuracy **on the test split** (reproduces the published-style inflation);
- `honest` — view chosen by **inner CV on the training split only** (leakage-free, nested);
- `fixed`  — each band over the full 0–4 s window, **no selection** (a no-cherry-picking floor).

This differs from your `timewindow_faithful` notebook (which used the 8 shipped bands and no LTSA). If `honest`
still lands near chance and `leaky` reproduces ~72–77%, the conclusion is locked across both implementations.

> Logging/artifacts mirror `liu2024_source_mat_sjepa_prelocal_augmented` (timestamped `print`→`run.log`, `RUN_ID`
> from CONFIG hash, `config.json`, CSV/JSON set) and the `summary.json` shape of your `timewindow_faithful` run so
> the numbers line up side by side. **Edit the single CONFIG cell (or apply a sweep) and re-run.** No torch/MNE needed.

# 1. Imports

In [1]:
import os, re, sys, json, random, hashlib, builtins, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import signal as sp_signal
from scipy.io import loadmat
from scipy import stats as sp_stats

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

# Riemannian geometry (your eeg-jepa env has pyriemann)
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.classification import FgMDM

warnings.filterwarnings("ignore")
print(f"numpy {np.__version__}")

numpy 2.4.3


# 2. CONFIG  *(edit this one cell, or apply a sweep JSON, then re-run)*

In [2]:
CONFIG = {
    # ---- identity ----
    "experiment_name": "twfb_dgfmdm_paper_faithful",
    "config_note":     "paper Table-3: 19 bands x 7 windows + real LTSA; leaky/honest/fixed",

    # ---- paths ----
    "data_root":    "../../liu2024_data/liu2024_figshare/sourcedata",
    "artifact_dir": "../../artifacts/liu2024_twfb_dgfmdm_paper_faithful",
    "subjects":     None,                 # None = all; or list like [1,2,7]
    "random_state": 2026,
    "sfreq_raw":    500,

    # ---- which modes to run (all reported together, like timewindow_faithful) ----
    "run_leaky":         True,
    "run_honest":        True,
    "run_perband_fixed": True,

    # ---- the paper's grid (Table 3) ----
    "time_window_starts_s": [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0],
    "time_window_len_s":    1.0,
    "freq_bands": [[8,12],[9,13],[10,14],[11,15],[12,16],[13,17],[14,18],[15,19],[16,20],
                   [17,21],[18,22],[19,23],[20,24],[21,25],[22,26],[23,27],[24,28],[25,29],[26,30]],

    # ---- onset / segmentation (matches your timewindow_faithful) ----
    "marker_channel_index":  32,
    "onset_marker_value":    2,
    "onset_plausible_range": [800, 1300],
    "onset_fallback_sample": 1003,
    "preroll_samples":       800,         # 1.6 s filter warm-up, trimmed off
    "notch_freq":            50.0,        # mains notch (set null to disable)
    "notch_Q":               6.0,
    "butter_order":          2,
    "filter_phase":          "zero",      # 'zero' (filtfilt) | 'causal' (one-pass)

    # ---- covariance ----
    "cov_estimator":      "scm",          # 'scm'|'oas'|'lwf'
    "cov_trace_normalize": True,
    "cov_shrinkage":      0.1,            # added as shrinkage*I after trace-norm (numerical stability)

    # ---- Step 5 classifier ----
    "twfb_classifier":  "lda_ltsa",       # 'lda_ltsa' (faithful: TangentSpace->LTSA->LDA) | 'fgmdm' (DGFMDM on covariances)
    "twfb_metric":      "riemann",        # tangent/FgMDM metric: 'riemann'|'logeuclid'

    # ---- LTSA (Step 4), used by 'lda_ltsa' ----
    "ltsa_pre_pca":     20,               # reduce 435-d tangent -> this before LTSA (stability; null to skip)
    "ltsa_n_components": 6,
    "ltsa_n_neighbors": 10,
    "ltsa_reg":         1e-2,

    # ---- cross-validation ----
    "cv_scheme":  "liu_repeated_holdout", # 'liu_repeated_holdout' (matches timewindow_faithful) | 'sjepa_5fold'
    "n_repeats":  10,
    "test_frac":  0.4,                    # 24 train / 16 test
    "n_splits":   5,                      # used for sjepa_5fold (32 train / 8 test)
    "inner_folds": 3,                     # inner CV for honest view selection
}

# ---- derived ----
DATA_ROOT = Path(CONFIG["data_root"])
FS        = CONFIG["sfreq_raw"]
WIN_LEN   = int(round(CONFIG["time_window_len_s"] * FS))
print(f"grid = {len(CONFIG['time_window_starts_s'])} windows x {len(CONFIG['freq_bands'])} bands = "
      f"{len(CONFIG['time_window_starts_s'])*len(CONFIG['freq_bands'])} views | clf={CONFIG['twfb_classifier']} | cv={CONFIG['cv_scheme']}")

grid = 7 windows x 19 bands = 133 views | clf=lda_ltsa | cv=liu_repeated_holdout


## 2.1 Logging & Artifact Init

In [3]:
def create_run_id():
    h = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{datetime.now().strftime('%Y%m%d_%H%M')}_{h}"
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _w(stream, t):
    try: stream.write(t)
    except UnicodeEncodeError:
        enc=getattr(stream,"encoding",None) or "utf-8"; stream.write(t.encode(enc,"replace").decode(enc,"replace"))
def _tprint(*a, **k):
    sep=k.pop("sep"," "); end=k.pop("end","\n"); k.pop("flush",False); k.pop("file",None)
    msg=sep.join(str(x) for x in a); lead=len(msg)-len(msg.lstrip("\n")); body=msg[lead:]
    if lead: _w(sys.stdout,"\n"*lead); _w(_LOG,"\n"*lead)
    if body:
        s=f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {body}"; _w(sys.stdout,s+end); _w(_LOG,s+end)
    else: _w(sys.stdout,end); _w(_LOG,end)
builtins.print=_tprint
random.seed(CONFIG["random_state"]); np.random.seed(CONFIG["random_state"])
with open(ARTIFACT_DIR/"config.json","w") as f: json.dump(CONFIG,f,indent=2,default=str)
print("="*70); print(f"Experiment: {CONFIG['experiment_name']}"); print(f"Note:       {CONFIG['config_note']}")
print(f"Run ID:     {RUN_ID}"); print(f"Artifacts:  {ARTIFACT_DIR}"); print("="*70)

[2026-06-17 13:16:34] ======================================================================
[2026-06-17 13:16:34] Experiment: twfb_dgfmdm_paper_faithful
[2026-06-17 13:16:34] Note:       paper Table-3: 19 bands x 7 windows + real LTSA; leaky/honest/fixed
[2026-06-17 13:16:34] Run ID:     20260617_1316_94a20d83
[2026-06-17 13:16:34] Artifacts:  ../../artifacts/liu2024_twfb_dgfmdm_paper_faithful/20260617_1316_94a20d83
[2026-06-17 13:16:34] ======================================================================


# 3. Channel Constants

In [4]:
SOURCE_EEG_NAMES_30 = ["Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4","FT7","FT8","Cz","C3","C4",
    "T3","T4","CPz","CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"]
CPZ_IDX=17; EEG_KEEP_IDX=[i for i in range(30) if i!=CPZ_IDX]
EEG_NAMES=[SOURCE_EEG_NAMES_30[i] for i in EEG_KEEP_IDX]; N_CHANS=len(EEG_KEEP_IDX)
MARKER_COL=CONFIG["marker_channel_index"]
print(f"N channels: {N_CHANS} (CPz dropped) | first 5 {EEG_NAMES[:5]}")

[2026-06-17 13:16:34] N channels: 29 (CPz dropped) | first 5 ['Fp1', 'Fp2', 'Fz', 'F3', 'F4']


# 4. Data Loading (raw 500 Hz + MI onset)

In [5]:
def subject_id_from_path(p):
    m=re.search(r"sub[-_ ]?(\d{1,2})", str(p), flags=re.IGNORECASE)
    return int(m.group(1)) if m else int(re.findall(r"\d+", Path(p).stem)[-1])

def find_mat_files(root):
    root=Path(root)
    if not root.exists(): raise FileNotFoundError(f"data root not found: {root}")
    return sorted(root.rglob("*.mat"))

def _load_mat_arrays(mat_path):
    mat=loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
    raw=mat.get("rawdata", mat.get("data")); lab=mat.get("labels", mat.get("label"))
    if raw is None or lab is None:
        for k,v in mat.items():
            if k.startswith("__"): continue
            if hasattr(v,"_fieldnames"):
                if raw is None and "rawdata" in v._fieldnames: raw=getattr(v,"rawdata")
                if lab is None and "label"   in v._fieldnames: lab=getattr(v,"label")
            elif raw is None and isinstance(v,np.ndarray) and v.ndim==3: raw=v
    raw=np.asarray(raw,dtype=np.float64)
    ax=next(a for a,s in enumerate(raw.shape) if s==40); raw=np.moveaxis(raw,ax,0)
    if raw.shape[1]!=33 and raw.shape[2]==33: raw=raw.transpose(0,2,1)
    assert raw.shape==(40,33,4000), f"shape {raw.shape}"
    y=np.asarray(lab,dtype=int).ravel()
    if set(np.unique(y)).issubset({1,2}): y=y-1
    assert len(y)==40 and set(np.unique(y)).issubset({0,1})
    return raw,y

def load_subject_raw(mat_path, cfg):
    """Return eeg500 (40,29,4000), onsets (40,), y (40,)."""
    raw,y=_load_mat_arrays(mat_path)
    eeg=raw[:,EEG_KEEP_IDX,:].astype(np.float64)
    marker=raw[:,MARKER_COL,:]
    lo,hi=cfg["onset_plausible_range"]; fb=cfg["onset_fallback_sample"]; mv=cfg["onset_marker_value"]
    onsets=[]
    for t in range(40):
        idx=np.where(np.isclose(marker[t],mv))[0]; idx=idx[(idx>=lo)&(idx<=hi)]
        onsets.append(int(idx[0]) if len(idx) else int(fb))
    return eeg, np.asarray(onsets,int), y
print("Loader defined.")

[2026-06-17 13:16:34] Loader defined.


# 5. Paper-Faithful TWFB-DGFMDM Pipeline (19 bands x 7 windows + LTSA)

In [6]:
# ---- filters ----
def _notch(x, fs, f0, q):
    b,a=sp_signal.iirnotch(f0/(0.5*fs), q); return sp_signal.filtfilt(b,a,x,axis=-1)
def _band(x, fs, lo, hi, order, phase):
    ny=0.5*fs; b,a=sp_signal.butter(order,[lo/ny,hi/ny],btype="band")
    return sp_signal.lfilter(b,a,x,axis=-1) if phase=="causal" else sp_signal.filtfilt(b,a,x,axis=-1)

def segment_view(eeg, onsets, win_start_s, band, cfg, win_len=None):
    fs=cfg["sfreq_raw"]; warm=cfg["preroll_samples"]; lo,hi=band
    L=win_len if win_len is not None else int(round(cfg["time_window_len_s"]*fs))
    n_t=eeg.shape[2]; out=np.zeros((len(onsets),N_CHANS,L))
    for i in range(len(onsets)):
        start=onsets[i]+int(round(win_start_s*fs)); w=warm; s0=start-w
        if s0<0: w=max(start,0); s0=0
        s1=min(start+L,n_t); seg=eeg[i,:,s0:s1]
        if cfg.get("notch_freq"): seg=_notch(seg,fs,cfg["notch_freq"],cfg["notch_Q"])
        seg=_band(seg,fs,lo,hi,cfg["butter_order"],cfg["filter_phase"])
        seg=seg[:,w:w+L] if seg.shape[1]>=w+L else seg[:,-L:]
        if seg.shape[1]<L: seg=np.pad(seg,((0,0),(0,L-seg.shape[1])),mode="edge")
        out[i]=seg
    return out

def covs_of(seg, cfg):
    C=Covariances(estimator=cfg["cov_estimator"]).transform(seg)
    if cfg.get("cov_trace_normalize", False):
        tr=np.trace(C,axis1=1,axis2=2)[:,None,None]; C=C/np.clip(tr,1e-12,None)
    s=cfg.get("cov_shrinkage",0.0)
    if s: C=(1-s)*C + s*np.eye(C.shape[1])[None]
    return C

def precompute_views(eeg, onsets, cfg):
    """dict[(wi,bi)] -> (40,29,29). Per-trial op = leakage-safe to precompute for all trials."""
    windows=cfg["time_window_starts_s"]; bands=cfg["freq_bands"]; views={}
    for wi,ws in enumerate(windows):
        for bi,bd in enumerate(bands):
            views[(wi,bi)]=covs_of(segment_view(eeg,onsets,ws,bd,cfg), cfg)
    keys=[(wi,bi) for wi in range(len(windows)) for bi in range(len(bands))]
    return views, keys

# ---- Step 4-5: TangentSpace -> (PCA) -> LTSA -> LDA   OR   FgMDM ----
def view_fit(cov_tr, y_tr, cfg):
    if cfg["twfb_classifier"]=="fgmdm":
        return {"kind":"fgmdm","clf":FgMDM(metric=cfg["twfb_metric"]).fit(cov_tr,y_tr)}
    ts=TangentSpace(metric=cfg["twfb_metric"]).fit(cov_tr); Z=ts.transform(cov_tr); pca=None
    if cfg.get("ltsa_pre_pca") and Z.shape[1]>cfg["ltsa_pre_pca"]:
        n=min(cfg["ltsa_pre_pca"], Z.shape[0]-1)
        pca=PCA(n_components=n, random_state=cfg["random_state"]).fit(Z); Z=pca.transform(Z)
    k=min(cfg["ltsa_n_neighbors"], Z.shape[0]-1)
    d=min(cfg["ltsa_n_components"], max(1,k-1), Z.shape[1])
    ltsa=None; Zr=Z
    try:
        ltsa=LocallyLinearEmbedding(method="ltsa", n_neighbors=k, n_components=d,
                                    reg=cfg["ltsa_reg"], eigen_solver="dense",
                                    random_state=cfg["random_state"]).fit(Z)
        Zr=ltsa.transform(Z)
    except Exception:
        ltsa=None; Zr=Z   # fall back to (pca'd) tangent features if LTSA is unstable on this fold
    lda=LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(Zr, y_tr)
    return {"kind":"lda_ltsa","ts":ts,"pca":pca,"ltsa":ltsa,"lda":lda}

def view_pred(obj, cov):
    if obj["kind"]=="fgmdm": return obj["clf"].predict(cov)
    Z=obj["ts"].transform(cov)
    if obj["pca"] is not None: Z=obj["pca"].transform(Z)
    if obj["ltsa"] is not None:
        try: Z=obj["ltsa"].transform(Z)
        except Exception: pass
    return obj["lda"].predict(Z)

# ---- view selection ----
def select_leaky(views, keys, tr, te, y, cfg):
    """LEAKY: pick the view with the best TEST balanced accuracy (oracle / data leakage)."""
    best,bs=keys[0],-1.0
    for k in keys:
        obj=view_fit(views[k][tr], y[tr], cfg)
        s=balanced_accuracy_score(y[te], view_pred(obj, views[k][te]))
        if s>bs: bs,best=s,k
    return best

def select_honest(views, keys, tr, y_tr, cfg):
    """HONEST: pick the view by inner-CV balanced accuracy on TRAIN only."""
    inner=StratifiedKFold(cfg["inner_folds"], shuffle=True, random_state=cfg["random_state"])
    best,bs=keys[0],-1.0
    for k in keys:
        cov=views[k][tr]; sc=[]
        for itr,iva in inner.split(cov, y_tr):
            if len(np.unique(y_tr[itr]))<2: continue
            sc.append(balanced_accuracy_score(y_tr[iva], view_pred(view_fit(cov[itr],y_tr[itr],cfg), cov[iva])))
        s=np.mean(sc) if sc else -1.0
        if s>bs: bs,best=s,k
    return best
print("Paper-faithful TWFB pipeline (TangentSpace -> LTSA -> LDA) defined.")

[2026-06-17 13:16:34] Paper-faithful TWFB pipeline (TangentSpace -> LTSA -> LDA) defined.


# 6. Per-Subject Runner (leaky / honest / fixed)

In [7]:
def subject_splits(y, cfg):
    if cfg["cv_scheme"]=="sjepa_5fold":
        sp=StratifiedKFold(cfg["n_splits"], shuffle=True, random_state=cfg["random_state"])
    else:
        sp=StratifiedShuffleSplit(n_splits=cfg["n_repeats"], test_size=cfg["test_frac"], random_state=cfg["random_state"])
    return list(sp.split(np.zeros(len(y)), y))

def run_subject(sid, eeg, onsets, y, cfg):
    views, keys = precompute_views(eeg, onsets, cfg)
    splits = subject_splits(y, cfg)
    leaky, honest = [], []
    fixed = {f"{lo}_{hi}": [] for (lo,hi) in cfg["freq_bands"]} if cfg["run_perband_fixed"] else {}
    # full-window covariances per band for the no-selection 'fixed' baseline
    fixed_cov = {}
    if cfg["run_perband_fixed"]:
        full_len=int(round((max(cfg["time_window_starts_s"])+cfg["time_window_len_s"])*0 + 4.0*cfg["sfreq_raw"]))
        for (lo,hi) in cfg["freq_bands"]:
            fixed_cov[f"{lo}_{hi}"]=covs_of(segment_view(eeg,onsets,0.0,[lo,hi],cfg,win_len=full_len), cfg)
    for (tr,te) in splits:
        if len(np.unique(y[tr]))<2 or len(np.unique(y[te]))<2: continue
        if cfg["run_leaky"]:
            k=select_leaky(views,keys,tr,te,y,cfg)
            leaky.append(balanced_accuracy_score(y[te], view_pred(view_fit(views[k][tr],y[tr],cfg), views[k][te])))
        if cfg["run_honest"]:
            k=select_honest(views,keys,tr,y[tr],cfg)
            honest.append(balanced_accuracy_score(y[te], view_pred(view_fit(views[k][tr],y[tr],cfg), views[k][te])))
        if cfg["run_perband_fixed"]:
            for bk,cov in fixed_cov.items():
                fixed[bk].append(balanced_accuracy_score(y[te], view_pred(view_fit(cov[tr],y[tr],cfg), cov[te])))
    row={"subject":int(sid),"n_trials":int(len(y))}
    if cfg["run_leaky"]:  row["twfb_leaky"]=float(np.mean(leaky))
    if cfg["run_honest"]: row["twfb_honest_acc"]=float(np.mean(honest)); row["twfb_honest_bacc"]=float(np.mean(honest))
    for bk,v in fixed.items(): row[f"fixed_{bk}"]=float(np.mean(v)) if v else float("nan")
    return row
print("Subject runner defined.")

[2026-06-17 13:16:34] Subject runner defined.


# 7. Run All Subjects

In [8]:
mat_files=find_mat_files(DATA_ROOT)
all_sids=sorted({subject_id_from_path(f) for f in mat_files})
SUBJECT_IDS=all_sids if not CONFIG["subjects"] else sorted(int(s) for s in CONFIG["subjects"])
sid_to_path={subject_id_from_path(f):f for f in mat_files if subject_id_from_path(f) in SUBJECT_IDS}
print(f"Found {len(mat_files)} .mat files | using {len(SUBJECT_IDS)} subjects | grid {len(CONFIG['time_window_starts_s'])}x{len(CONFIG['freq_bands'])}")
print("="*70)
SUBJECT_ROWS=[]
for sid in SUBJECT_IDS:
    p=sid_to_path.get(sid)
    if p is None: print(f"  Sub {sid:02d}: no file"); continue
    try:
        eeg,onsets,y=load_subject_raw(p, CONFIG)
        row=run_subject(sid, eeg, onsets, y, CONFIG)
    except Exception as exc:
        print(f"  Sub {sid:02d}: error — {exc}"); continue
    SUBJECT_ROWS.append(row)
    msg=f"  Sub {sid:02d}:"
    if "twfb_leaky" in row:       msg+=f" leaky={row['twfb_leaky']*100:5.1f}%"
    if "twfb_honest_bacc" in row: msg+=f" honest={row['twfb_honest_bacc']*100:5.1f}%"
    print(msg)
print("="*70); print(f"Done. {len(SUBJECT_ROWS)} subjects.")

[2026-06-17 13:16:34] Found 50 .mat files | using 50 subjects | grid 7x19
[2026-06-17 13:16:34] ======================================================================
[2026-06-17 13:34:23]   Sub 01: leaky= 76.2% honest= 51.2%
[2026-06-17 13:52:11]   Sub 02: leaky= 79.4% honest= 48.1%
[2026-06-17 14:09:57]   Sub 03: leaky= 80.6% honest= 52.5%
[2026-06-17 14:27:45]   Sub 04: leaky= 78.1% honest= 56.9%
[2026-06-17 14:45:22]   Sub 05: leaky= 80.6% honest= 49.4%
[2026-06-17 15:03:10]   Sub 06: leaky= 76.2% honest= 43.1%
[2026-06-17 15:21:00]   Sub 07: leaky= 81.9% honest= 53.8%
[2026-06-17 15:38:41]   Sub 08: leaky= 82.5% honest= 46.2%
[2026-06-17 15:56:26]   Sub 09: leaky= 81.9% honest= 50.0%
[2026-06-17 16:14:11]   Sub 10: leaky= 80.0% honest= 56.2%
[2026-06-17 16:31:54]   Sub 11: leaky= 83.1% honest= 48.8%
[2026-06-17 16:49:39]   Sub 12: leaky= 78.8% honest= 52.5%
[2026-06-17 17:07:03]   Sub 13: leaky= 77.5% honest= 48.1%
[2026-06-17 17:24:46]   Sub 14: leaky= 81.2% honest= 56.9%
[2026-0

# 8. Aggregate, Save Artifacts, Leaky-vs-Honest Plot

In [9]:
if not SUBJECT_ROWS: raise RuntimeError("No subjects completed — check data_root.")
df=pd.DataFrame(SUBJECT_ROWS)
df.to_csv(ARTIFACT_DIR/"subject_results.csv", index=False)

def mean_std(col):
    v=df[col].dropna().values; return [float(v.mean()), float(v.std(ddof=1))]
summary={"config":CONFIG, "n_subjects":int(len(df))}
S={}
if CONFIG["run_leaky"]:  S["twfb_leaky"]=mean_std("twfb_leaky")
if CONFIG["run_honest"]: S["twfb_honest_acc"]=mean_std("twfb_honest_acc"); S["twfb_honest_bacc"]=mean_std("twfb_honest_bacc")
if CONFIG["run_perband_fixed"]:
    fb_cols=[c for c in df.columns if c.startswith("fixed_")]
    fb_means={c:float(df[c].mean()) for c in fb_cols}
    best=max(fb_means, key=fb_means.get)
    S["best_fixed_band"]=[best, fb_means[best]]; S["mean_fixed_band"]=float(np.mean(list(fb_means.values())))
summary["summary"]=S

# significance of honest vs chance (per-subject)
if CONFIG["run_honest"]:
    h=df["twfb_honest_bacc"].dropna().values
    t,pt=sp_stats.ttest_1samp(h,0.5)
    try: w,pw=sp_stats.wilcoxon(h-0.5)
    except Exception: pw=float("nan")
    summary["honest_vs_chance"]={"mean":float(h.mean()),"t_p":float(pt),"wilcoxon_p":float(pw),
                                 "n_above_chance":int((h>0.5).sum()),"n_subjects":int(len(h))}

with open(ARTIFACT_DIR/"summary.json","w") as f: json.dump(summary,f,indent=2,default=str)
with open(ARTIFACT_DIR/"global_metrics.json","w") as f: json.dump({"experiment_name":CONFIG["experiment_name"],**S},f,indent=2)
with open(ARTIFACT_DIR/"run_metadata.json","w") as f:
    json.dump({"run_id":RUN_ID,"artifact_dir":str(ARTIFACT_DIR),"config":CONFIG,"n_subjects":int(len(df))},f,indent=2,default=str)

print("="*70); print(f"GLOBAL — paper-faithful TWFB-DGFMDM [{CONFIG['twfb_classifier']} | grid {len(CONFIG['time_window_starts_s'])}x{len(CONFIG['freq_bands'])}]")
for k,v in S.items():
    if isinstance(v,list) and len(v)==2 and isinstance(v[0],float):
        print(f"  {k:18s} {v[0]*100:5.2f}% ± {v[1]*100:4.2f}")
    else:
        print(f"  {k:18s} {v}")
if "honest_vs_chance" in summary:
    hv=summary["honest_vs_chance"]; print(f"  honest vs chance: wilcoxon p={hv['wilcoxon_p']:.3g} | {hv['n_above_chance']}/{hv['n_subjects']} subj > 0.5")
if CONFIG["run_leaky"] and CONFIG["run_honest"]:
    print(f"  >>> LEAKAGE GAP: leaky {S['twfb_leaky'][0]*100:.1f}% -> honest {S['twfb_honest_bacc'][0]*100:.1f}% "
          f"= {(S['twfb_leaky'][0]-S['twfb_honest_bacc'][0])*100:.1f} pts")
print(f"  Artifacts: {ARTIFACT_DIR}"); print("="*70)

# comparison plot
fig,ax=plt.subplots(figsize=(6,4)); labels=[]; means=[]; errs=[]; colors=[]
if CONFIG["run_leaky"]:  labels.append("TWFB leaky\n(selection sees test)"); means.append(S["twfb_leaky"][0]); errs.append(S["twfb_leaky"][1]); colors.append("#c0392b")
if CONFIG["run_perband_fixed"]: labels.append(f"best fixed band\n({S['best_fixed_band'][0].replace('fixed_','')} Hz)"); means.append(S["best_fixed_band"][1]); errs.append(0); colors.append("#b8860b")
if CONFIG["run_honest"]: labels.append("TWFB honest\n(nested CV)"); means.append(S["twfb_honest_bacc"][0]); errs.append(S["twfb_honest_bacc"][1]); colors.append("#2e7d4f")
x=np.arange(len(labels)); ax.bar(x,[m*100 for m in means],yerr=[e*100 for e in errs],capsize=4,color=colors,alpha=0.85)
ax.axhline(50,color="k",ls="--",lw=1,label="chance"); ax.axhline(72.21,color="purple",ls=":",lw=1.2,label="published 72.21%")
ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8); ax.set_ylabel("balanced accuracy (%)"); ax.set_ylim(40,85)
ax.legend(fontsize=8); ax.set_title("Paper-faithful TWFB-DGFMDM: leaky vs honest")
for xi,m in zip(x,means): ax.text(xi,m*100+1,f"{m*100:.1f}%",ha="center",fontsize=9)
plt.tight_layout(); plt.savefig(ARTIFACT_DIR/"leaky_vs_honest.png",dpi=200,bbox_inches="tight"); plt.close(fig)
print("Saved subject_results.csv, summary.json, global_metrics.json, run_metadata.json, leaky_vs_honest.png")

[2026-06-18 04:03:15] ======================================================================
[2026-06-18 04:03:15] GLOBAL — paper-faithful TWFB-DGFMDM [lda_ltsa | grid 7x19]
[2026-06-18 04:03:15]   twfb_leaky         80.34% ± 3.00
[2026-06-18 04:03:15]   twfb_honest_acc    51.66% ± 5.21
[2026-06-18 04:03:15]   twfb_honest_bacc   51.66% ± 5.21
[2026-06-18 04:03:15]   best_fixed_band    ['fixed_9_13', 0.5092500000000001]
[2026-06-18 04:03:15]   mean_fixed_band    0.4962105263157895
[2026-06-18 04:03:15]   honest vs chance: wilcoxon p=0.047 | 26/50 subj > 0.5
[2026-06-18 04:03:15]   >>> LEAKAGE GAP: leaky 80.3% -> honest 51.7% = 28.7 pts
[2026-06-18 04:03:15]   Artifacts: ../../artifacts/liu2024_twfb_dgfmdm_paper_faithful/20260617_1316_94a20d83
[2026-06-18 04:03:15] ======================================================================
[2026-06-18 04:03:15] Saved subject_results.csv, summary.json, global_metrics.json, run_metadata.json, leaky_vs_honest.png
